### 微调

#### 指令数据集准备

In [2]:
import json
import random
from collections import defaultdict
from pathlib import Path


# =========================
# 配置区
# =========================
INPUT_PATH = "/root/autodl-tmp/atd/data/人工标注/最终结果/merged_dataset_Type_flattened.json"        # 原始数据
OUTPUT_PATH = "/root/autodl-tmp/atd/data/人工标注/最终结果/wsd_train.json"
SAMPLE_PER_LABEL = 200
RANDOM_SEED = 42
DROP_IF_NOT_ENOUGH = False   # True: 不足200条则丢弃该朝代


# =========================
# system prompt
# =========================
SYSTEM_PROMPT = """你是一位古汉语专家, 能够正确从译文中找出与古文中指定汉字直接对应的词语。请严格遵循以下步骤：

### 一、识别分析

- 仔细阅读古文和对应译文
- 定位指定汉字在古文中的所有出现位置
- 分析每个出现位置的上下文语境
- 区分实词与虚词, 采用不同的提取策略：
  - 实词（名词、动词、形容词等）：从现代汉语译文中提取对应该词的意思
  - 虚词（介词、连词、助词等）：直接说明语法功能
  - 专有名词：直接输出专有名词

### 二、提取对应

- 实词提取：
  - 对古文字符每个出现位置, 从译文中提取对应的现代汉语
  - 确保释义完整正确, 完全来自译文
- 虚词处理：
  - 直接说明字符语法功能(如"表示...")
- 专有名词：
  - 若字符对应词汇为专有名词, 则直接提取专有名词
- 特殊情形处理：
  - 如果译文中没有直接对应词, 必须根据译文进行对词语进行补充

### 三、输出规范

- ['释义1', ...]
- 输出为一个该字符的出现顺序排列的释义列表
- 释义内容必须在译文中出现或为译文扩充内容
- 严格按JSON格式输出

### 四、重要原则

1. 当无法直接提取释义时, 需根据译文进行增补
2. 提取的意思应简短并完全准确（长度为1-4个字）
3. 禁止提取古文内容
4. 若译文无相应释义, 回复"unknown"
"""


# =========================
# 构造 input
# =========================
def build_input(sample):
    text = sample["text"]
    word = sample["word"]

    return f'古文：“{text}”\n字符：{word}'


# =========================
# 主流程
# =========================
def main():
    random.seed(RANDOM_SEED)

    # 1. 读取数据
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"原始数据量: {len(data)}")

    # 2. 按 label 分组
    grouped = defaultdict(list)
    for item in data:
        label = item.get("label", "unknown")
        grouped[label].append(item)

    print(f"朝代数量: {len(grouped)}")

    # 3. 采样
    sampled_data = []

    for label, items in grouped.items():
        print(f"{label}: {len(items)} 条")

        if len(items) < SAMPLE_PER_LABEL:
            if DROP_IF_NOT_ENOUGH:
                print(f"⚠️ 跳过 {label}")
                continue
            else:
                sampled = items
        else:
            sampled = random.sample(items, SAMPLE_PER_LABEL)

        sampled_data.extend(sampled)

    print(f"采样后数据量: {len(sampled_data)}")

    # 4. 构建 LoRA 数据
    lora_dataset = []

    for item in sampled_data:
        output = item.get("final_option_id")

        # 过滤无效数据
        if not output:
            continue

        example = {
            "instruction": SYSTEM_PROMPT,
            "input": build_input(item),
            "output": output
        }

        lora_dataset.append(example)

    print(f"最终可用数据量: {len(lora_dataset)}")

    # 5. 保存
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(lora_dataset, f, ensure_ascii=False, indent=2)

    print(f"✅ 已保存到: {OUTPUT_PATH}")


# =========================
# 入口
# =========================
if __name__ == "__main__":
    main()

原始数据量: 24000
朝代数量: 12
宋: 2000 条
南朝宋: 2000 条
西晋: 2000 条
后晋: 2000 条
元: 2000 条
西汉: 2000 条
唐: 2000 条
北朝齐: 2000 条
清: 2000 条
南朝梁: 2000 条
东汉: 2000 条
明: 2000 条
采样后数据量: 2400
最终可用数据量: 2400
✅ 已保存到: /root/autodl-tmp/atd/data/人工标注/最终结果/wsd_train.json


修改 `/root/LLaMA-Factory/data/dataset_info.json` 文件: 添加一个指令数据集的描述，如下
```json
  "wsd_train": {
    "file_name": "/root/autodl-tmp/atd/data/人工标注/最终结果/wsd_train.json"
  },
````
键将作为后续配置文件的数据集名称使用

#### 微调参数文件准备

当前保存在 `/root/LLaMA-Factory/examples/train_lora/Qwen3-1.5B-Instruct.yaml`

```yaml
### model
# 手动下载的需要放绝对路径，需要自行修改
# 修改下面这一行/root/autodl-tmp/models/Qwen3-1.5B-Instruct的路径就行
model_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct
trust_remote_code: true


### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.1
lora_target: all
deepspeed: /root/LLaMA-Factory/examples/deepspeed/ds_z0_config.json  # choices: [ds_z0_config.json, ds_z2_config.json, ds_z3_config.json]

### dataset
# dataset: identity,alpaca_en_demo
dataset: wsd_train

# template 不成功的清按详情参照https://github.com/hiyouga/LlamaFactory/blob/main/README_zh.md#%E8%AE%AD%E7%BB%83%E6%96%B9%E6%B3%95abs
# 下一行为当前需要使用的模板，{模型类别}:{template}
# qwen3:qwen3; qwen2.5:qwen; gemma:gemma; llama2:llama2; llama3:llama3; gpt2:gpt_oss; deepseek-distill: deepseekr1
template: qwen
cutoff_len: 2048
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16
dataloader_num_workers: 4


### output
# 这里是训练完成后的权重和部分中间过程的内容，需要自行修改
output_dir: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none  # choices: [none, wandb, tensorboard, swanlab, mlflow]


### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 2
learning_rate: 1.0e-4
num_train_epochs: 30.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
resume_from_checkpoint: null
```

#### 训练命令

`/root/LLaMA-Factory/examples/train_lora/1-Qwen3-1.5B-Instruct.yaml`就修改成训练的配置文件路径即可

FORCE_TORCHRUN=1 llamafactory-cli train /root/LLaMA-Factory/examples/train_lora/1-Qwen3-1.5B-Instruct.yaml

训练完成的权重保存在 `/root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora`

#### 合并模型和适配器

配置文件: `LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml`

```yaml
### Note: DO NOT use quantized model or quantization_bit when merging lora adapters

### model
# 原始模型的导出路径，需要自行修改
model_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct
# lora权重的导出路径，需要自行修改
adapter_name_or_path: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora
# 不同模型的模板，需要自行修改
template: qwen
trust_remote_code: true

### export
# 合并模型的导出路径，需要自行修改
export_dir: /root/autodl-tmp/models/Qwen3-1.5B-Instruct/lora_models
export_size: 2
export_device: auto  # choices: [cpu, auto]
export_legacy_format: false
```

运行命令

`/root/LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml`前面**合并模型和适配器**的配置文件，需要自行修改

llamafactory-cli export /root/LLaMA-Factory/examples/merge_lora/Qwen3-1.5B-Instruct.yaml

#### 启动模型

##### xinference (本次使用)

如果需要单独的模型，则需要注册-配置-启动一条龙